### Usage Metrics
We can now define the methodology of our investigation. First we will consider observable actions that can result in a possession ending. This can be a shot on goal, an inaccurate pass, a player being dispossessed, an unsuccessful dribble or a miscontrol. The sum of these will define a player's total number of Possession Ending Actions (PEA). This data was obtainable over the 25/26 season, with the exception of miscontrols. Therefore, a player's PEA will be calculated as such:
$$\text{PEA} = \text{Shots} + \text{Inaccurate Passes} + \text{Dispossessed} + \text{Unsuccessful Dribbles} $$
Since not all players across the league play an even amount of minutes, the PEA values are normalised for minutes played to create a player's PEA/90:
$$\text{PEA / 90} = \frac{\text{PEA}}{\text{Minutes Played}} \times 90 $$

Since this investigation aims to evaluate how well players use the ball given their share of PEA, we need a metric to define output. Shot Creating Actions (SCA) are the perfect metric, defined as the two offensive actions immediately proceeding a shot. This is a really useful metric since it credits secondary actions like an interception or take on which leads to a pass leading to a shot, therefore giving a better view of a player's offensive contribution. However, once again, this data was not available to us, so we will instead quantify a player's offensive output using Chances Created (CC). This is defined as a pass directly leading to a shot, comprised of two metrics. Key passes are passes that lead to a shot without scoring and assists are passes leading to a shot that results in a goal. Therefore Chances Created are calculated as:
$$\text{CC} = \text{Key Passes} + \text{Assists}$$
Once again we can then calculate the chances created per 90 minutes:
$$\text{CC / 90} = \frac{\text{CC}}{\text{Minutes Played}} \times 90$$

The limitation with the current framework is that it is hard to compare PEA / 90 and CC / 90 across all teams in the league due to possession. Players on teams with greater average possession generally have more opportunity to become involved in possession ending actions and chance creation. For example, Rayan Cherki might have higher PEA / 90 and CC / 90 stats than a player like Harry Wilson since it is likely that Manchester City will on average see a much higher share of possesion than Fulham so these stats are not immediately informative. We will therefore assume that player involvement scales linearly with team possesion and normalise so that each team has a neutral 50% possession share, adjusting the stats as such:
$$\text{PEA / 90}_{\text{adj}} = \text{PEA / 90} \times \frac{50}{\text{Team Possession}}$$
$$\text{CC / 90}_{\text{adj}} = \text{CC / 90} \times \frac{50}{\text{Team Possession}}$$

At this point we will be able to make a fair comparison of all players across the league and identify how CC / 90 varies with PEA / 90 across different groups of positions.

To investigate a player's role within their own team, we want to look at how much of a team's possession flow through a single player and how much of the team's total output can be attributed to that player. Since we will not be considering goalkeepers in this analysis, the total team PEA and CC will be calculated as the sum of all elgible outfield players:
$$\text{Team PEA} = \sum{\text{Total Player PEA}}$$
$$\text{Team CC} = \sum{\text{Total Player CC}}$$

Then we can define a new metric, a player's Usage Rate, which is the proportion of team possessions that end with that player:
$$\text{Usage Rate} = \frac{\text{Player PEA}}{\text{Team PEA}}$$
We can also find a player's Chance Creation Involvement as:
$$\text{Chance Creation Involvement} = \frac{\text{Player CC}}{\text{Team CC}}$$

Plotting these two metrics against each other, we will be able to identify a player's importance within their team and compare it to their output, creating one final metric, Relative Creation Efficiency (RCE):
$$\text{RCE} = \frac{\text{CC Involvement}}{\text{Usage Rate}}$$

From this analysis we will be able to measure the standout players across the league and also the most important players within specific teams.

In [2]:
# Import the cleaned data of players and teams
import pandas as pd

players_df = pd.read_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/cleaned/players_cleaned.csv")
teams_df = pd.read_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/cleaned/teams_cleaned.csv")

In [3]:
# 1: Create columns for PEA and PEA/90
players_df["PEA"] = players_df["totalShots"] + players_df["inaccuratePasses"] + players_df["dispossessed"] + players_df["unsuccessfulDribbles"]
players_df["PEA/90"] = (players_df["PEA"] / players_df["minutesPlayed"]) * 90

# Round PEA/90 
players_df["PEA/90"] = players_df["PEA/90"].round(2)

# Check 
print(players_df[["playerName","minutesPlayed","totalShots","inaccuratePasses", "dispossessed", "unsuccessfulDribbles","PEA","PEA/90"]].head())

        playerName  minutesPlayed  totalShots  inaccuratePasses  dispossessed  \
0  Bruno Fernandes           3066          85               355            30   
1      Declan Rice           3099          40               271            18   
2  Bruno Guimarães           2455          42               201            46   
3            Rodri           1513          16               160            11   
4      Jérémy Doku           1784          39               132            49   

   unsuccessfulDribbles  PEA  PEA/90  
0                    21  491   14.41  
1                    14  343    9.96  
2                    30  319   11.69  
3                     8  195   11.60  
4                    64  284   14.33  


In [4]:
# 2: Create columns for CC and CC/90
players_df["CC"] = players_df["keyPasses"] + players_df["assists"]
players_df["CC/90"] = (players_df["CC"] / players_df["minutesPlayed"]) * 90

# Round CC/90
players_df["CC/90"] = players_df["CC/90"].round(2)

# Check
print(players_df[["playerName","minutesPlayed","keyPasses","assists","CC","CC/90"]].head())

        playerName  minutesPlayed  keyPasses  assists   CC  CC/90
0  Bruno Fernandes           3066        137       21  158   4.64
1      Declan Rice           3099         64        5   69   2.00
2  Bruno Guimarães           2455         46        5   51   1.87
3            Rodri           1513         25        0   25   1.49
4      Jérémy Doku           1784         60        5   65   3.28


In [5]:
# 3: Create columns for possession adjusted PEA/90 and CC/90
players_df["adjPEA/90"] = (players_df["PEA/90"] * (50 / players_df["averageBallPossession"])).round(2)
players_df["adjCC/90"] = (players_df["CC/90"] * (50 / players_df["averageBallPossession"])).round(2)

# Check
print(players_df[["playerName","averageBallPossession","PEA/90","adjPEA/90","CC/90","adjCC/90"]].head())

        playerName  averageBallPossession  PEA/90  adjPEA/90  CC/90  adjCC/90
0  Bruno Fernandes                  51.79   14.41      13.91   4.64      4.48
1      Declan Rice                  56.13    9.96       8.87   2.00      1.78
2  Bruno Guimarães                  52.61   11.69      11.11   1.87      1.78
3            Rodri                  60.50   11.60       9.59   1.49      1.23
4      Jérémy Doku                  60.50   14.33      11.84   3.28      2.71


In [6]:
# 4: Create columns in the teams dataframe for total PEA and CC
# Obtain team total PEA and CC in a new df
team_totals = (players_df.groupby("teamId")[["PEA","CC"]].sum().reset_index())

# Rename the columns
team_totals = team_totals.rename(columns = {
    "PEA" : "totalPEA",
    "CC" : "totalCC"
})

# Merge the team total PEA and CC into the teams_df
teams_df = teams_df.merge(team_totals,
                         on = "teamId",
                         how = "left")

# Check
print(teams_df.head())

                 teamName  teamId  goalsScored  averageBallPossession  \
0                 Arsenal      42           71                  56.13   
1            Leeds United      34           49                  45.74   
2  Brighton & Hove Albion      30           52                  54.08   
3               Brentford      50           55                  47.61   
4              Sunderland      41           42                  44.84   

   totalPEA  totalCC  
0      3614      458  
1      3578      352  
2      3682      403  
3      3707      331  
4      3339      292  


In [44]:
# 5: Calculate Usage Rates and Chance Creation Involvement 
# Remove existing team totals if they are already present (due to re-running the script)
players_df = players_df.drop(columns=["totalPEA", "totalCC"],
                             errors="ignore")

# Merge the team totals into the players_dfplayers_df = players_df.merge(
players_df = players_df.merge(teams_df[["teamId", "totalPEA", "totalCC"]],
                              on="teamId",
                              how="left")

# Calculate Usage rate
players_df["usageRate"] = (players_df["PEA"] / players_df["totalPEA"]).round(3)

# Calculate Chance Creation Involvement
players_df["chanceInvolvement"] = (players_df["CC"] / players_df["totalCC"]).round(3)

# Check
print(players_df[["playerName", "usageRate", "chanceInvolvement"]].head())

# Check that the usage rate and chance involvement roughly sums to 1 for each team allowing for slight deviation since usage rate and chance involvement were rounded to 3 decimal places
print(players_df.groupby("teamId")["usageRate"].sum())
print(players_df.groupby("teamId")["chanceInvolvement"].sum())

        playerName  usageRate  chanceInvolvement
0  Bruno Fernandes      0.131              0.315
1      Declan Rice      0.095              0.151
2  Bruno Guimarães      0.088              0.131
3            Rodri      0.047              0.046
4      Jérémy Doku      0.069              0.119
teamId
3     0.998
6     1.002
7     1.001
14    1.001
17    1.001
30    1.001
33    0.998
34    1.003
35    0.998
37    1.000
38    1.000
39    1.001
40    0.999
41    1.000
42    1.001
43    0.999
44    0.998
48    0.999
50    1.003
60    1.001
Name: usageRate, dtype: float64
teamId
3     1.000
6     1.000
7     0.999
14    1.001
17    1.004
30    0.997
33    1.003
34    1.004
35    1.001
37    1.000
38    0.998
39    0.996
40    0.998
41    0.998
42    1.002
43    1.002
44    1.000
48    1.002
50    1.000
60    1.001
Name: chanceInvolvement, dtype: float64


In [66]:
# 6: Create one final metrics which is Relative Creation Efficiency (REC)
players_df["RCE"] = (players_df["chanceInvolvement"] / players_df["usageRate"]).round(3)

# Convert inf values to nan
import numpy as np
players_df["RCE"] = players_df["RCE"].replace([np.inf, -np.inf], np.nan)

# Fill nan values
players_df["RCE"] = players_df["RCE"].fillna(0)

In [68]:
# 6: Export the final sheet 
# Define the list of columns
feature_cols = ["playerName","playerId","teamName","teamId","primaryPosition","positionGroup","minutesPlayed",
               "PEA","PEA/90","adjPEA/90",
               "CC", "CC/90", "adjCC/90",
               "usageRate", "chanceInvolvement", "RCE",
               "assists","expectedAssists","goals","expectedGoals",
               "marketValue"]
# Create a new df
player_usage_data = players_df[feature_cols].copy()

# Export the final player usage metrics df as well as a master copy of the full data
player_usage_data.to_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/player_usage_data.csv", index = False)
players_df.to_csv("/Users/Seth/Desktop/Football Analyses/Premier League Usage Rates/data/players_master.csv", index = False)